# Day 6 — 인증 및 미디어 처리 기초

### 2.4 인증 모듈 단독 테스트

서버 없이 인증 로직만 먼저 테스트합니다.

참고: 주피터 노트북은 내부적으로 이벤트 루프가 이미 실행 중이므로 asyncio.run()을 사용하면 에러가 발생합니다. 노트북에서는 셀 최상위에서 await를 직접 사용할 수 있습니다.


In [13]:
from app.auth import verify_api_key
from fastapi import HTTPException

# 테스트 1: 올바른 키
try:
    user = await verify_api_key(x_api_key="test1-key-001")  # *your code* — 올바른 키
    print(f"✅ 인증 성공: {user}")
except HTTPException as e:
    print(f"❌ 인증 실패: {e.detail}")

✅ 인증 성공: {'user_id': 11, 'name': 'test-user', 'role': 'admin'}


### 3.3 FastAPI의 UploadFile 기본 사용법

In [2]:
# UploadFile의 핵심 속성 확인 (실행 가능)
print("UploadFile의 핵심:")
print("  file.filename     → 파일 이름 (예: 'cat.png')")
print("  file.content_type → MIME 타입 (예: 'image/png')")
print("  await file.read() → 파일 내용 (bytes)")
print()
print("실제 서버 코드에서의 사용은 섹션 6에서 직접 구현합니다.")

UploadFile의 핵심:
  file.filename     → 파일 이름 (예: 'cat.png')
  file.content_type → MIME 타입 (예: 'image/png')
  await file.read() → 파일 내용 (bytes)

실제 서버 코드에서의 사용은 섹션 6에서 직접 구현합니다.


## 4. 파일 업로드와 안전장치: 크기 제한, 이미지 리사이징

### 4.2 안전장치 구현

In [3]:
%%writefile app/image_utils.py
"""
Day 6 - 이미지 업로드 안전장치 + 전처리
"""
from fastapi import UploadFile, HTTPException
from PIL import Image
import io

# 허용 설정
ALLOWED_TYPES = {"image/png", "image/jpeg", "image/jpg"}
MAX_FILE_SIZE = 5 * 1024 * 1024  # 5MB                     # *your code* — 최대 파일 크기


async def validate_and_read_image(
    file: UploadFile,
    max_size: int = MAX_FILE_SIZE,
    target_size: tuple = (28, 28),
) -> Image.Image:
    """
    업로드된 파일을 검증하고, PIL 이미지로 반환합니다.

    검증 순서:
      1. 파일 타입 검증 → 허용된 형식(PNG, JPEG)만 통과
      2. 파일 크기 검증 → 5MB 이하만 통과
      3. 이미지 디코딩 검증 → 실제로 열 수 있는 이미지만 통과
      4. 리사이징 + 그레이스케일 변환 → 모델 입력 크기에 맞춤
    """

    # ─── 1. 파일 타입 검증 ─────────────────────────
    # content_type은 클라이언트가 보낸 MIME 타입입니다.
    # .exe를 .png로 위장해도 content_type이 다르므로 차단됩니다.
    if file.content_type not in ALLOWED_TYPES:               # *your code* — 타입 체크
        raise HTTPException(
            status_code=400,
            detail=f"지원하지 않는 파일 형식입니다: {file.content_type}. "
                   f"허용 형식: {ALLOWED_TYPES}",
        )

    # ─── 2. 파일 크기 검증 ─────────────────────────
    # 파일 전체를 읽어서 크기를 확인합니다.
    # 이 시점에서 파일 내용이 메모리에 올라옵니다.
    contents = await file.read()
    if len(contents) > max_size:                             # *your code* — 크기 체크
        raise HTTPException(
            status_code=400,
            detail=f"파일 크기가 {max_size // (1024*1024)}MB를 초과합니다. "
                   f"현재: {len(contents) / (1024*1024):.1f}MB",
        )

    # ─── 3. 이미지 디코딩 검증 ─────────────────────
    # content_type이 image/png여도 파일 내용이 실제로 이미지가 아닐 수 있습니다.
    # PIL로 열어보면서 확인합니다.
    try:
        image = Image.open(io.BytesIO(contents))
    except Exception:
        raise HTTPException(
            status_code=400,
            detail="이미지를 읽을 수 없습니다. 파일이 손상되었을 수 있습니다.",
        )

    # ─── 4. 리사이징 + 그레이스케일 변환 ──────────────
    # 어떤 크기의 이미지가 들어와도 모델 입력에 맞게 변환합니다.
    image = image.convert("L").resize(target_size)           # *your code* — 그레이스케일 + 리사이즈

    return image

Writing app/image_utils.py


### 4.3 안전장치 단독 테스트

In [2]:
from PIL import Image
import io

# 테스트 이미지 생성 (28x28 그레이스케일)
# 테스트 이미지 생성 (28x28 그레이스케일)
test_img = Image.new("L", (100, 100), color=128)   # 100x100 회색 이미지, L: 그레이스케일 모드(8비트, 0=검은색 ~ 255=흰색)
buffer = io.BytesIO()   # 파일 대신 메모리에서 읽기/쓰기 가능한 가상 파일
test_img.save(buffer, format="PNG")
test_bytes = buffer.getvalue()  # buffer의 전체 내용을 bytes 객체로 추출(헤더+이미지 데이터 포함)

print(f"테스트 이미지 크기: {len(test_bytes)} bytes")
print(f"테스트 이미지 해상도: {test_img.size}")

테스트 이미지 크기: 120 bytes
테스트 이미지 해상도: (100, 100)


In [6]:
# 리사이징 테스트
img = Image.open(io.BytesIO(test_bytes))
img_resized = img.convert("L").resize((28, 28))
print(f"변환 전: {img.size} → 변환 후: {img_resized.size}")

변환 전: (100, 100) → 변환 후: (28, 28)


In [7]:
# 잘못된 파일 테스트
try:
    Image.open(io.BytesIO(b"this is not an image"))
except Exception as e:
    print(f"디코딩 실패: {type(e).__name__}")
# → 이런 경우 validate_and_read_image()가 400 에러를 반환합니다.

디코딩 실패: UnidentifiedImageError


## 6. 실습: 이미지를 업로드하면 분류 결과를 반환하는 API 만들기

### 6.1 통합 API 서버 코드

In [15]:
%%writefile app/image_api.py
"""
Day 6 - 이미지 분류 API (인증 + 파일 업로드 + MNIST 모델)
"""
import asyncio
from concurrent.futures import ThreadPoolExecutor

import torch
from fastapi import FastAPI, UploadFile, File, Depends, HTTPException
from torchvision import transforms

from app.model_utils import load_model, predict, CLASS_NAMES
from app.auth import verify_api_key
from app.image_utils import validate_and_read_image
from app.logger_config import setup_logger
from app.error_handlers import register_error_handlers
from app.middleware import RequestLoggingMiddleware


# ===== 설정 =====
logger = setup_logger("image_api")

app = FastAPI(
    title="Image Classification API",
    description="이미지를 업로드하면 숫자(0~9)를 분류하는 API (인증 필요)",
    version="1.0.0",
)

app.add_middleware(RequestLoggingMiddleware)
register_error_handlers(app)

executor = ThreadPoolExecutor(max_workers=4, thread_name_prefix="image")

# ===== 모델 로드 =====
MODEL_PATH = "models/mnist_state_dict.pth"
model = None

# 전처리 파이프라인 (PIL → Tensor)
img_transform = transforms.Compose([               # *your code* — transforms.Compose 구성
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])


@app.on_event("startup")
async def startup():
    global model
    logger.info("MNIST 모델 로드 중...")
    model = load_model(MODEL_PATH)
    logger.info("모델 로드 완료")


def run_inference(tensor: torch.Tensor) -> dict:
    if model is None:
        raise RuntimeError("모델이 로드되지 않았습니다")
    return predict(model, tensor)


# ===== 엔드포인트 =====

@app.get("/health", tags=["System"])
async def health_check():
    return {"status": "healthy" if model else "loading"}


@app.post("/predict/image", tags=["Inference"])
async def predict_image(
    file: UploadFile = File(..., description="분류할 이미지 (PNG, JPEG)"),
    user: str = Depends(verify_api_key),                     # *your code* — 인증 적용
):
    """
    이미지를 업로드하면 숫자(0~9)를 분류합니다.
    X-API-Key 헤더에 유효한 API Key가 필요합니다.
    """
    logger.info(f"추론 요청 — 사용자: {user}, 파일: {file.filename}")

    # 1. 파일 검증 + 이미지 로드 (28x28 그레이스케일로 변환)
    image = await validate_and_read_image(file, target_size=(28, 28))  # *your code* — 안전장치 적용

    # 2. 텐서 변환
    tensor = img_transform(image).unsqueeze(0)   # (1, 1, 28, 28)

    # 3. 비동기 추론
    try:
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(executor, run_inference, tensor)
    except Exception as e:
        logger.error(f"추론 실패: {e}")
        raise HTTPException(status_code=500, detail=f"추론 실패: {str(e)}")

    logger.info(f"추론 완료 — 결과: {result['predicted_class']}, 확신도: {result['confidence']:.2f}")

    return {
        "success": True,
        "predicted_class": result["predicted_class"],
        "confidence": round(result["confidence"], 4),
        "user": user,
    }

Overwriting app/image_api.py


### 6.2 서버 실행

In [11]:
# ⚠️ 이전 섹션에서 서버를 실행했다면, 반드시 커널을 재시작하세요.
#    "Address already in use" → Kernel → Restart Kernel 후 이 셀부터 실행

import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_server():
    uvicorn.run("app.image_api:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("✅ 서버 시작됨 — http://localhost:8000/docs 에서 Swagger UI를 확인하세요.")

# terminal 실행
# uvicorn app.image_api:app --host 0.0.0.0 --port 8000 --reload

INFO:     Started server process [18308]
INFO:     Waiting for application startup.


2026-04-03 14:13:06 INFO     [image_api] MNIST 모델 로드 중...


INFO:image_api:MNIST 모델 로드 중...


2026-04-03 14:13:06 INFO     [image_api] 모델 로드 완료


INFO:image_api:모델 로드 완료
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ 서버 시작됨 — http://localhost:8000/docs 에서 Swagger UI를 확인하세요.


### 6.3 테스트 1: 인증 없이 요청 → 401

In [23]:
import requests

# API Key 없이 요청
response = requests.post(
    "http://localhost:8000/predict/image",
    files={"file": ("test.png", b"fake image data", "image/png")},
    # headers 없음 → 인증 실패
)

print(f"상태 코드: {response.status_code}")   # 401
print(f"응답: {response.json()}")

상태 코드: 401
응답: {'detail': 'X-API-Key header is missing'}


### 6.4 테스트 2: 잘못된 키 → 401

In [24]:
response = requests.post(
    "http://localhost:8000/predict/image",
    files={"file": ("test.png", b"fake image data", "image/png")},
    headers={"X-API-Key": "wrong-key"},                      # *your code* — 잘못된 키
)

print(f"상태 코드: {response.status_code}")   # 401
print(f"응답: {response.json()}")

상태 코드: 401
응답: {'detail': 'Invalid API key'}


### 6.5 테스트 3: 올바른 키 + MNIST 이미지 → 성공

In [25]:
from torchvision import datasets
from PIL import Image
import io

# MNIST 테스트 이미지 가져오기
test_dataset = datasets.MNIST(root="data", train=False, download=True)
test_image, test_label = test_dataset[0]   # 첫 번째 테스트 이미지

# PIL 이미지 → bytes 변환
buf = io.BytesIO()
test_image.save(buf, format="PNG")
image_bytes = buf.getvalue()

print(f"테스트 이미지 정답: {test_label}")

# API 호출
response = requests.post(
    "http://localhost:8000/predict/image",
    files={"file": ("digit.png", image_bytes, "image/png")},
    headers={"X-API-Key": "test1-key-001"},                   # *your code* — 올바른 키
)

print(f"상태 코드: {response.status_code}")   # 200
result = response.json()
print(f"예측 결과: {result}")

테스트 이미지 정답: 7
상태 코드: 200
예측 결과: {'success': True, 'predicted_class': '7', 'confidence': 1.0, 'user': {'user_id': 11, 'name': 'test-user', 'role': 'admin'}}


### 6.6 테스트 4: 잘못된 파일 형식 → 400

In [26]:
response = requests.post(
    "http://localhost:8000/predict/image",
    files={"file": ("test.txt", b"this is not an image", "text/plain")},
    headers={"X-API-Key": "test1-key-001"},
)

print(f"상태 코드: {response.status_code}")   # 400
print(f"응답: {response.json()}")

상태 코드: 400
응답: {'detail': "지원하지 않는 파일 형식입니다: text/plain. 허용 형식: {'image/png', 'image/jpeg', 'image/jpg'}"}


### 6.7 테스트 5: 여러 이미지 연속 테스트

In [28]:
import requests
from torchvision import datasets
from PIL import Image
import io

test_dataset = datasets.MNIST(root="data", train=False, download=True)

print("=== 연속 추론 테스트 (5장) ===\n")

for i in range(5):
    img, label = test_dataset[i]

    buf = io.BytesIO()
    img.save(buf, format="PNG")

    resp = requests.post(
        "http://localhost:8000/predict/image",
        files={"file": (f"digit_{i}.png", buf.getvalue(), "image/png")},
        headers={"X-API-Key": "묘"},
    )

    r = resp.json()
    predicted = r.get("predicted_class", "?")
    confidence = r.get("confidence", 0)
    match = "✅" if str(label) == str(predicted) else "❌"

    print(f"  이미지 {i}: 정답={label}, 예측={predicted}, 확신도={confidence:.4f} {match}")

=== 연속 추론 테스트 (5장) ===

  이미지 0: 정답=7, 예측=7, 확신도=1.0000 ✅
  이미지 1: 정답=2, 예측=2, 확신도=1.0000 ✅
  이미지 2: 정답=1, 예측=1, 확신도=0.9999 ✅
  이미지 3: 정답=0, 예측=0, 확신도=1.0000 ✅
  이미지 4: 정답=4, 예측=4, 확신도=1.0000 ✅
